# 04 - Validation, Diagnostics, and the Bugs That Mattered

This notebook is mostly about **how a working pipeline can look broken and a
broken one can look working**. Every check here exists because a cheaper check
gave the wrong answer.

## 4.1 Validate adapters - against the base model and at real prompt length

Two independent checks, because they fail differently:

1. **Held-out loss vs the UNADAPTED base** - not epoch-to-epoch (blind to epoch 1).
2. **Behavioural divergence** - same probes from adapted and unadapted weights.

Critically, probes are rendered through the **real agent templates**
(`config.LEGAL_PROMPT` etc.) with real retrieved context.

In [ ]:
!.venv\Scripts\python.exe finetune\validate_adapters.py --max-new-tokens 400

### The validation harness itself produced two wrong verdicts

**Wrong verdict 1 - passed broken adapters.** Original probes were ~50-token bare
questions. Output looked merely *terse*; the degeneration only appeared at ~1400
tokens. Short probes cannot reveal long-context failure.

**Wrong verdict 2 - failed working adapters.** The first fix wrapped probes in an
*improvised* scaffold (`'You are a legal expert... citing specific Articles'`)
rather than the real `LEGAL_PROMPT`. The adapters are **format-sensitive by
construction** - they are trained on targets shaped like those templates. Under
the improvised prompt the legal adapter answered in **36 words** where its base
used 213, reading as a severe regression. Under the real prompt, the same
adapter produced **165-687 words of correct IRAC citing the right article**,
while the base model **fabricated a citation** (`Article 2(1)(b)`) or abstained.

**Also: the pass criteria measure whether the adapter CHANGED the model, not
whether it improved it.** Divergence and loss gain both passed an adapter that
was answering with a third of its base model's words. The report now prints word
ratio and citation counts next to the verdict.

In [ ]:
# Final measured validation, real agent prompts:
#
#   role        probe tok  divergence  words T/B    ratio   citations T/B
#   legal       1674       98.3%       165 vs 7     23.5x   1.0 vs 0.0
#   news         243       93.3%       214 vs 72     3.0x   0.0 vs 0.0
#   general_qa   262       89.7%       120 vs 59     2.0x   0.0 vs 0.0
#
# The legal base model's '7 words' is the abstention sentence:
#   'Insufficient authoritative support -- recommend expert review.'
# So at a real prompt the UNADAPTED model declines while the adapter answers and
# cites Article 43 / Article 8. That is a behavioural change in abstention, which
# this paper treats as a first-class outcome - and the benchmark, not two probes,
# is what settles whether it is correct.
import json
r = json.load(open('finetune/adapter_validation.json', encoding='utf-8'))
for e in r['roles']:
    print(f"{e['role']:11s} {e['status']:7s} divergence={e['mean_diff_ratio']:.3f} "
          f"words {e['tuned_mean_words']:.0f}/{e['base_mean_words']:.0f} "
          f"gain_from_base={e.get('gain_from_base')}")

## 4.2 Direct expert probe - bypassing the graph

`/chat` cannot answer *'is this adapter usable'*, because the answer has passed
through the aggregator, validator and response nodes - any of which could be the
degrading step. This calls `LegalAgent` directly with the real retrieval stack
and the real prompt.

In [ ]:
!.venv\Scripts\python.exe probe_legal_expert.py

Measured: at a **1292-token** production prompt the tuned legal expert produced
**687 words** of correct IRAC citing **Article 6(3)** (correct for high-risk
classification), while the base produced 165 words citing **Article 2(1)(b)** -
a fabricated citation with a fabricated quote.

## 4.3 THE decisive bug - coordination nodes were loading an expert's adapter

The experts were fine all along. The word-salad came from the **aggregator and
response nodes**, which are supposed to run the general expert's base weights
with the adapter **disabled**.

```
[local_models] loading role=general_qa base=granite-3.1-2b-instruct adapter=yes   <-- WRONG
aggregator model: granite-3.1-2b-instruct  adapter=.../adapters/general_qa
```

**Cause - one argument.** `LocalChatModel.__init__` called
`get_loaded_model(self.role)`, but `self.role` was *already resolved*:
`resolve_role('aggregator') -> 'general_qa'`. `get_loaded_model` derives the
adapter decision from the role it is handed, so it answered for the **expert**
instead of the coordination node.

Every coordination node - aggregator, validator, response, router, planner,
memory, QueryAnalyzer - ran with the general expert's LoRA attached.

**Why it survived a fix:** `get_loaded_model` *was* corrected to consult the
requested role, and a regression test asserted exactly that - by calling
`get_loaded_model('aggregator')` directly. It never went through the constructor,
which is the only place the defect lived.

**Fix:** pass the *requested* role. `get_loaded_model(role)`, not `self.role`.

In [ ]:
# The probe that exposed it: hold the model fixed, vary only how many expert
# answers are fed in, so the prompt walks from ~1500 to ~2100 tokens.
!.venv\Scripts\python.exe probe_aggregator.py

### Before and after, same probe

| input | before (`adapter=yes`) | after (`adapter=no`) |
|---|---|---|
| 1 expert, 1546 tok | word-salad, 1024 tok, **hit_cap=True**, 159 s | **228 words**, 331 tok, hit_cap=False, **42 s** |
| 2 experts, 1690 tok | word-salad, 1024 tok, hit_cap=True, 165 s | 233 words, 352 tok, hit_cap=False, 43 s |
| 3 experts, 2102 tok | word-salad, 1024 tok, hit_cap=True, 165 s | 214 words, 312 tok, hit_cap=False, 37 s |

Degeneration at **every** prompt size, so it was never length-related - which is
why a 1024-token retrain did not help.

**It was also the runtime problem.** The broken node never emitted a stop token,
so it generated the full 1024-token budget every time (`hit_cap=True` on all
three). ~4x faster once fixed. Every other coordination node was equally
afflicted - the `response` node took 240 s in one run for the same reason.

## 4.4 Missing repetition penalty (a real, separate defect)

`transformers` defaults `repetition_penalty` to **1.0 - none**. Ollama, which
every pre-pivot run used, applies **1.1** by default. Moving generation from
Ollama to raw `transformers` silently removed repetition control the earlier
experiments always had, and 2-3B models asked for 1024 tokens collapse into
loops (`'Under Article Article Article...'` x80).

Fixed via `config.LOCAL_REPETITION_PENALTY = 1.1`, chosen to **match Ollama's
default** so decoding stays comparable with the pre-pivot runs rather than
introducing a second uncontrolled difference.

Also now recorded per generation: **`hit_token_cap`** - hitting the cap exactly
is the signature of a model that never emitted its stop token.

## 4.5 Diagnostic traps - read this before debugging anything here

| trap | consequence |
|---|---|
| Reusing one `session_id` across probes | the memory agent loads persisted history, so each probe echoes the previous answer. A freshly retrained adapter reproduced the OLD adapter's output almost verbatim and looked unchanged. `benchmark.py` is safe - unique id per (arm, mode, query, repeat). |
| Short probes | hide long-context degeneration entirely |
| Improvised prompt scaffolds | adapters are format-sensitive; measures a condition they never face |
| Trusting the loss curve | 46-72% loss reduction alongside unusable generation |
| Epoch-to-epoch curves | blind to everything learned in epoch 1; made a working adapter look inert |
| Bare `python` | system interpreter lacks peft/accelerate/bitsandbytes; models fail to load with a misleading message |
| Ollama down | retrieval silently degrades to BM25-only and still completes |